# 00 - Configuração e autenticação

Configuração inicial, feita uma única vez por cada membro da equipa. Cria a
árvore de pastas, verifica as credenciais e confirma que todos os IDs de
datasets do Earth Engine indicados no plano existem.

**Antes de correr**

1. Copiar `.env.example` para `.env` e preencher `EE_PROJECT` (o projeto Google
   Cloud registado no Earth Engine) e, para o GloFAS, `CDS_API_KEY`.
2. Num terminal: `earthengine authenticate` (abre o navegador uma vez).

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import os
import pandas as pd
from ete import config

cfg = config.load_config()
print("Raiz do repositório:", ROOT)
print("EE_PROJECT         :", cfg["gee"]["project"] or "NÃO DEFINIDO -> editar o .env")
print("CDS_API_KEY        :", "definida" if os.getenv("CDS_API_KEY") or (Path.home() / ".cdsapirc").exists() else "NÃO DEFINIDA (só é necessária para o GloFAS)")

Raiz do repositório: /home/miocha/all_information/Projecto-ETE
EE_PROJECT         : ete-incomati-6162
CDS_API_KEY        : definida


## 1. Criar a árvore de pastas

In [2]:
for key in cfg["paths"]:
    if key != "manifest":
        config.path(cfg, key)
print("\n".join(sorted(v for k, v in cfg["paths"].items() if k != "manifest")))

data/interim
data/metadata
data/processed
data/raw/aoi
data/raw/discharge/glofas
data/raw/exposure/buildings
data/raw/exposure/osm
data/raw/exposure/population
data/raw/hydromet/era5_land
data/raw/hydromet/smap
data/raw/labels/ems_unosat
data/raw/labels/flood_events
data/raw/labels/jrc_monthly_history
data/raw/labels/sen1floods11
data/raw/land_cover/dynamic_world
data/raw/optical/landsat
data/raw/optical/sentinel2
data/raw/precipitation/chirps
data/raw/precipitation/imerg
data/raw/radar/sentinel1
data/raw/static/hydrography
data/raw/static/land_cover
data/raw/static/surface_water
data/raw/static/terrain


## 2. Earth Engine

In [ ]:
import ee
from ete import gee

gee.init(cfg)
print("Earth Engine OK:", ee.Number(1).add(1).getInfo() == 2)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine OK: True


: 

## 3. Confirmar os IDs e as versões dos datasets

O plano pede que cada ID seja confirmado no catálogo antes de ser usado. As
colunas `first`/`last` mostram a cobertura temporal real de cada coleção (a nível
global, não apenas sobre Marracuene).

In [ ]:
ds = cfg["datasets"]
ids = {f"landsat {k}": v["id"] for k, v in ds["landsat"]["sensors"].items()}
ids |= {name: ds[name]["id"] for name in ["sentinel1", "sentinel2", "chirps", "imerg", "era5_land", "smap", "jrc_monthly", "dynamic_world"]}
ids |= {f"static {k}": v for k, v in ds["static"].items()}
ids |= {"worldpop": ds["exposure"]["worldpop"], "open_buildings": ds["exposure"]["open_buildings"]}
ids |= {"district": cfg["aoi"]["district_asset"], "basins": f"{cfg['aoi']['basins_asset_root']}/hybas_{cfg['aoi']['subbasin_level']}"}

rows = []
for name, asset_id in ids.items():
    try:
        rows.append({"dataset": name, **gee.describe_asset(asset_id), "status": "ok"})
    except Exception as err:
        rows.append({"dataset": name, "id": asset_id, "status": f"PROBLEMA: {str(err)[:80]}"})
catalogue = pd.DataFrame(rows)
catalogue.to_csv(config.path(cfg, "metadata", "gee_catalogue_check.csv"), index=False)
catalogue

Uma linha com `PROBLEMA` significa que o ID mudou ou foi substituído (por
exemplo, uma nova versão do SMAP). Corrigir o ID em `config/config.yaml`, e não
nos notebooks.